In [1]:
from utils.data import ProteinDataset, ProteinPairDataset, pair_collate_fn
import torch as pt


data = pt.load(f'/home/burger/bioinfo/project/data/engineered_data.pt')
datalib = ProteinDataset(data)
pdb2idx = [(data[2][i], i) for i in range(len(data[2]))] # pdb name -> idx
pdb2idx = dict(pdb2idx)

/tmp/ipykernel_144042/3103729711.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = pt.load(f'/home/burger/bioinfo/project/data/engineered_data.pt')


In [2]:
from torch.utils.data import DataLoader



pair_dataset = ProteinPairDataset(datalib, '/home/burger/bioinfo/project/data/tmalign.out', pdb2idx)
loader = DataLoader(pair_dataset, batch_size=256, shuffle=False, collate_fn=pair_collate_fn, num_workers=6)

In [3]:
import torch as pt
import torch.nn as nn
from torch_geometric.data import Data
import torch_geometric.nn as gnn


class EmbeddingBlock(nn.Module):
    def __init__(self, out_channels:int=256):
        super().__init__()
        self.emb = nn.Embedding(num_embeddings=20, embedding_dim=out_channels)

    def forward(self, data):
        seq, graph, _ = data
        node_attr, edge_index, edge_len = graph.x, graph.edge_index, graph.edge_attr
        N = node_attr.size(0)
        # edge_index:[2, N-1+num_nho] edge_attr:[N-1+num_nho]
        tai_idx = pt.stack((edge_index[0, :N-1], edge_index[1, :N-1]), dim=0)
        nho_idx = pt.stack((edge_index[0, N-1:], edge_index[1, N-1:]), dim=0)
        edge_idx = pt.cat((tai_idx, tai_idx.flip(0), nho_idx, nho_idx.flip(0)), dim=0)
        tai_len = edge_len[:N-1]
        nho_len = edge_len[N-1:]
        edge_len = pt.cat((tai_len, tai_len.flip(0), nho_len, nho_len.flip(0)), dim=0)
        # edge_attr : 键长, is_peptide, direction, is_hbond, 肽键：±1 氢键：0
        is_peptide = pt.cat((pt.ones_like(tai_len), pt.ones_like(tai_len), pt.zeros_like(nho_len), pt.zeros_like(nho_len)), dim=0)
        is_hbond = 1 - is_peptide
        direction = pt.cat((pt.ones_like(tai_len), -1*pt.ones_like(tai_len), pt.zeros_like(nho_len), pt.zeros_like(nho_len)), dim=0)
        edge_attr = pt.stack((edge_len, is_peptide, is_hbond, direction), dim=0)
        node_emb = self.emb(seq)
        node_emb = pt.cat((node_emb, node_attr.unsqueeze(-1)), dim=1)
        return Data(x=node_emb, edge_index=edge_idx, edge_attr=edge_attr)

In [4]:
class ProteinGCN(nn.Module):
    def __init__(self, embed_dim:int=256, hidden_channels:int=256, num_layers:int=3):
        super().__init__()
        self.emb = EmbeddingBlock(out_channels=embed_dim)
        self.gcn = gnn.GCN(embed_dim, hidden_channels, num_layers, embed_dim,)
        self.shared = nn.Sequential(nn.Linear(4*embed_dim, embed_dim), nn.ReLU(),)
        self.tm_head = nn.Linear(embed_dim, 1)
        self.seq_head = nn.Linear(embed_dim, 1)

    def embbed(self, seq):
        self.eval()
        with pt.no_grad():
            emb = self.emb(seq)
        return emb

    def forward(self, data):
        seq_graph_i, seq_graph_j = data
        seq_i, graph_i = seq_graph_i
        seq_j, graph_j = seq_graph_j
        emb_i, emb_j = self.emb(seq_i), self.emb(seq_j)
        x_i, edge_index_i, edge_attr_i, batch_i = graph_i.x, graph_i.edge_index, graph_i.edge_attr, graph_i.batch
        x_j, edge_index_j, edge_attr_j, batch_j = graph_j.x, graph_j.edge_index, graph_j.edge_attr, graph_j.batch
        x_i = self.gcn(pt.cat((x_i, emb_i), dim=0), edge_index_i, edge_attr_i, batch_i)
        x_j = self.gcn(pt.cat((x_j, emb_j), dim=0), edge_index_j, edge_attr_j, batch_j)
        feature = pt.cat([x_i, x_j, pt.abs(x_i - x_j), x_i * x_j], dim=-1)
        shared_feat = self.shared(feature)
        tm_score = pt.sigmoid(self.tm_head(shared_feat)).squeeze(-1)
        seq_score = self.seq_head(shared_feat).squeeze(-1)
        return tm_score, seq_score

In [14]:
from sklearn.model_selection import train_test_split
import numpy as np
from torch_geometric.data import Batch

def pair_collate_fn(batch):
    seqs_graphs_i, seqs_graphs_j, scores = zip(*batch)
    seqs_i, graphs_i = zip(*seqs_graphs_i)
    seqs_j, graphs_j = zip(*seqs_graphs_j)
    seqs_i = pt.from_numpy(list(seqs_i))
    seqs_j = pt.from_numpy(list(seqs_j))
    batch_graph_i = Batch.from_data_list(graphs_i)
    batch_graph_j = Batch.from_data_list(graphs_j)
    return (seqs_i, batch_graph_i), (seqs_j, batch_graph_j), pt.stack(scores, dim=0)

gpu = 6
data_map = np.arange(len(pair_dataset), dtype=np.int64)
train_map, test_map = train_test_split(data_map, test_size=10240, random_state=42)
train_set = ProteinPairDataset(pair_dataset, mapping=train_map)
test_set = ProteinPairDataset(pair_dataset, mapping=test_map)
batch_size = 256
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=pair_collate_fn, drop_last=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, collate_fn=pair_collate_fn, num_workers=6)

In [6]:
import torch.nn.functional as F


prot_model = ProteinGCN().to(gpu)
criterion = nn.SmoothL1Loss()
learning_rate = 1e-3
optimizer = pt.optim.AdamW(prot_model.parameters(), lr=learning_rate)
num_epochs = 20

In [15]:
import logging
lamda = 0.1


for epoch in range(num_epochs):
    train_loss = []
    prot_model.train()
    for batch in train_loader:
        data_i, data_j, label = batch
        seqs_i, graphs_i = data_i
        seqs_j, graphs_j = data_j
        print('graph:', type(graphs_i), type(graphs_j))
        print('seq:', type(seqs_i), type(seqs_j))
        print('label:', type(label))
        graphs_i, seqs_i, graphs_j, seqs_j, label = \
            graphs_i.to(gpu), seqs_i.to(gpu), graphs_j.to(gpu), seqs_j.to(gpu), label.to(gpu)
        
        output = prot_model(((graphs_i, seqs_i), (graphs_j, seqs_j)))
        tm_score, seq_score = output
        tm_loss = criterion(tm_score, label[:, 0])
        seq_loss = criterion(seq_score, label[:, 1])
        loss = tm_loss + lamda * seq_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss.append(loss.item())
    logging.info(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {pt.tensor(train_loss).mean():.4f}')

TypeError: expected np.ndarray (got list)

In [9]:
a = (0, 1)
x, y = zip(*a)
print(x)
print(y)

TypeError: 'int' object is not iterable